# 検証つき蒸留の試し回し（life #45・docs/verified-distill-plan.md §5 の 1・Colab L4 用）

目的は **計算の量を測ること** だけ。この回の学習の成績は使わない。

1. 教師（Qwen3.8-27B UD-Q5_K_M・llama.cpp b11836・思考 OFF・温度 0.7・並列 8）に学習用の 100 問を 2 回ずつ答えさせ、`evaluation/distill/verify.py` で照合する
2. 残った答えで 200 件まで（Unsloth の公式 Gemma 4 E4B Text ノート 2362e6c の形で 4bit QLoRA・r 32・alpha 64・学習率 1e-4・2 エポック）学習する
3. 16bit で結合 → llama.cpp b11836 で bf16 の GGUF → Q4_K_M。学習していない E4B も同じ手順で Q4_K_M にする（対照）
4. 守りの問 40 問を、対照と学習後の両方で流す。学習文と llama-server の会話の形が一致するかも確かめる

各段の時間を `STAGE_JSON` に出す。ランタイムは **L4**。

In [ ]:
# 準備: リポジトリ・GPU・llama.cpp（配布版の実行ファイルと、変換用のソース）
!git clone -q --depth 1 -b research/verified-distill https://github.com/Maaa2005/local-rag-agent.git /content/repo
!git -C /content/repo log -1 --format='repo HEAD %H %s'
!git clone -q --depth 1 -b b11836 https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /content | tail -1

In [ ]:
%%capture
# Unsloth の公式ノート（unslothai/notebooks 2362e6c・Gemma4_(E4B)-Text）の入れ方そのまま
# 先に、動いている環境の numpy の版を控える（入れ直しで上がると unsloth が止まるため、最後に戻す）
import numpy as _np
_NP_LOADED = _np.__version__
import os, re
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install "huggingface_hub>=1.5.0,<2.0"
import importlib.metadata as _md
try: _tv = tuple(int(x) for x in _md.version("torch").split("+")[0].split(".")[:2])
except Exception: _tv = (2, 11)
_torchcodec = {(2, 11): "torchcodec>=0.11.0", (2, 10): "torchcodec==0.10.*", (2, 9): "torchcodec==0.9.*", (2, 8): "torchcodec==0.7.*", (2, 7): "torchcodec==0.5.*"}.get(_tv, "torchcodec>=0.12.0" if _tv >= (2, 12) else "torchcodec")
!pip install "{_torchcodec}"
!pip install --no-deps --upgrade timm
!pip install -q "numpy=={_NP_LOADED}"
# 変換（convert_hf_to_gguf.py）は別の環境で動かす。gguf 0.19 は numpy 2.2.6 以上を求め、学習の環境と合わないため
!pip install -q uv
!uv venv -q /content/conv-env
!uv pip install -q --python /content/conv-env/bin/python torch --index-url https://download.pytorch.org/whl/cpu
!uv pip install -q --python /content/conv-env/bin/python /content/llama.cpp/gguf-py "transformers==5.5.0" safetensors sentencepiece protobuf

In [ ]:
import json, os, pathlib, re, signal, subprocess, sys, time, urllib.request, hashlib, statistics
from concurrent.futures import ThreadPoolExecutor
sys.path.insert(0, "/content/repo")
from evaluation.distill.gen import make
from evaluation.distill import verify, guard as guardmod

OUTP = pathlib.Path("/content/trial"); OUTP.mkdir(exist_ok=True)
STAGES = []
T0 = time.time()
def stage(name, result, **nums):
    rec = {"stage": name, "result": result, "minutes": round((time.time() - T0) / 60, 1), **nums}
    STAGES.append(rec)
    print("STAGE_JSON " + json.dumps(rec, ensure_ascii=False))

GPU = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
if "L4" not in GPU:
    raise RuntimeError(f"L4 でない: {GPU}")
LLAMA_TAG = "b11836"
pre = pathlib.Path("/content/llama-pre"); pre.mkdir(exist_ok=True)
base = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}"
for a in (f"llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz", f"cudart-llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"):
    r = subprocess.run(["bash", "-c", f"set -o pipefail; curl -sfL {base}/{a} | tar xz -C {pre}"], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{a} が取れない: {r.stderr[-300:]}")
SERVER = str(sorted(pre.rglob("llama-server"))[0])
QUANTIZE = str(sorted(pre.rglob("llama-quantize"))[0])
LLAMA_ENV = dict(os.environ, LD_LIBRARY_PATH=":".join(sorted({str(p.parent) for p in pre.rglob("*.so*")}) + [os.environ.get("LD_LIBRARY_PATH", "")]))
REPO_SHA = subprocess.run(["git", "-C", "/content/repo", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
TRIAL_SEED = 777
ROWS = make.generate("train", 100, TRIAL_SEED)
GUARD = [json.loads(l) for l in open("/content/repo/evaluation/distill/guard/guard.jsonl", encoding="utf-8")]
print(GPU, "/ repo", REPO_SHA, "/ 試しの問", len(ROWS), "/ 守りの問", len(GUARD))
stage("準備", "pass", gpu=GPU, repo=REPO_SHA)

In [ ]:
# 教師の重み（進み具合の表示があるので別のセル）
from huggingface_hub import hf_hub_download
TEACHER = hf_hub_download("unsloth/Qwen3.8-27B-GGUF", "Qwen3.8-27B-UD-Q5_K_M.gguf",
                          revision="4ca720788d1e01f1bff70c033e0d0028fd02e502", local_dir="/content/gguf")

In [ ]:
stage("教師の重み", "pass", gib=round(os.path.getsize(TEACHER) / 2**30, 2))

def launch(argv, log):
    f = open(log, "w")
    return subprocess.Popen(argv, stdout=f, stderr=subprocess.STDOUT, start_new_session=True, env=LLAMA_ENV)

def kill(p):
    if p is None:
        return
    try:
        os.killpg(p.pid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    p.wait()

def wait_up(port, p, limit=900):
    t = time.time()
    while time.time() - t < limit and p.poll() is None:
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=3); return True
        except Exception:
            time.sleep(3)
    return False

def chat(port, messages, temperature, max_tokens, stream=False):
    body = {"model": "m", "messages": messages, "temperature": temperature, "max_tokens": max_tokens, "stream": stream}
    req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/chat/completions", json.dumps(body).encode(),
                                 {"Content-Type": "application/json"})
    t = time.time()
    if not stream:
        res = json.loads(urllib.request.urlopen(req, timeout=900).read())
        ch = res["choices"][0]
        return {"content": ch["message"].get("content") or "", "reasoning": len(ch["message"].get("reasoning_content") or ""),
                "finish": ch.get("finish_reason"), "usage": res.get("usage"), "seconds": time.time() - t}
    first, text, finish = None, "", None
    with urllib.request.urlopen(req, timeout=900) as r:
        for line in r:
            line = line.decode().strip()
            if not line.startswith("data:") or line == "data: [DONE]":
                continue
            d = json.loads(line[5:])
            if d.get("choices"):
                delta = d["choices"][0].get("delta", {}).get("content") or ""
                if delta and first is None:
                    first = time.time() - t
                text += delta
                finish = d["choices"][0].get("finish_reason") or finish
    return {"content": text, "first": first, "seconds": time.time() - t, "finish": finish}

# 教師: 並列 8・1 枠 4,096
tp = launch([SERVER, "-m", TEACHER, "-ngl", "999", "-c", str(8 * 4096), "-np", "8", "--jinja", "--reasoning", "off",
             "--host", "127.0.0.1", "--port", "8001"], OUTP / "teacher.log")
if not wait_up(8001, tp):
    print("\n".join(open(OUTP / "teacher.log").read().splitlines()[-30:]))
    raise RuntimeError("教師が起動しない")
jobs = [(r, k) for r in ROWS for k in range(2)]
t = time.time()
def ask(job):
    r, k = job
    out = chat(8001, r["messages"], 0.7, 512)
    ok, why = verify.accept(r, out["content"], out["finish"], (out["usage"] or {}).get("completion_tokens"))
    return {"id": r["id"], "k": k, "category": r["category"], "ok": ok, "why": why, "answer": out["content"],
            "tokens": (out["usage"] or {}).get("completion_tokens"), "prompt_tokens": (out["usage"] or {}).get("prompt_tokens"),
            "reasoning": out["reasoning"]}
with ThreadPoolExecutor(8) as ex:
    TEACH = list(ex.map(ask, jobs))
teach_sec = time.time() - t
kill(tp)
(OUTP / "teacher.jsonl").write_text("".join(json.dumps(x, ensure_ascii=False) + "\n" for x in TEACH), encoding="utf-8")
from collections import Counter, defaultdict
by_q = defaultdict(list)
for x in TEACH:
    if x["ok"]:
        by_q[x["id"]].append(x["answer"])
kept = {qid: verify.dedupe(ans) for qid, ans in by_q.items()}
if sum(len(v) for v in kept.values()) < 100:   # 足りなければ 100 問を足す（1 回だけ）
    extra = make.generate("train", 100, TRIAL_SEED + 1)
    tp = launch([SERVER, "-m", TEACHER, "-ngl", "999", "-c", str(8 * 4096), "-np", "8", "--jinja", "--reasoning", "off",
                 "--host", "127.0.0.1", "--port", "8001"], OUTP / "teacher2.log")
    wait_up(8001, tp)
    with ThreadPoolExecutor(8) as ex:
        more = list(ex.map(ask, [(r, k) for r in extra for k in range(2)]))
    kill(tp)
    TEACH += more; ROWS += extra
    for x in more:
        if x["ok"]:
            by_q[x["id"]].append(x["answer"])
    kept = {qid: verify.dedupe(ans) for qid, ans in by_q.items()}
cat_q = Counter(r["category"] for r in ROWS)
cat_ok = Counter(r["category"] for r in ROWS if r["id"] in kept)
summary = {"answers": len(TEACH), "seconds": round(teach_sec), "sec_per_answer": round(teach_sec / len(TEACH), 2),
           "answers_ok": sum(x["ok"] for x in TEACH), "questions_ok": len(kept), "examples": sum(len(v) for v in kept.values()),
           "why": dict(Counter(x["why"] for x in TEACH)), "by_category_q": {c: f"{cat_ok[c]}/{cat_q[c]}" for c in cat_q},
           "tokens_median": statistics.median([x["tokens"] or 0 for x in TEACH]),
           "prompt_tokens_max": max(x["prompt_tokens"] or 0 for x in TEACH), "reasoning_any": any(x["reasoning"] for x in TEACH)}
print("TEACHER_JSON " + json.dumps(summary, ensure_ascii=False))
os.remove(TEACHER)   # 場所を空ける
stage("教師の答え", "pass", **{k: summary[k] for k in ("seconds", "answers_ok", "questions_ok", "examples")})

In [ ]:
# 学習（公式ノートの形。値は docs/verified-distill-plan.md §4・試し回しは学習率 1e-4 の 1 本だけ）
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

ROW = {r["id"]: r for r in ROWS}
examples = [{"messages": ROW[q]["messages"] + [{"role": "assistant", "content": a}]} for q, ans in kept.items() for a in ans][:200]
t = time.time()
model, tokenizer = FastModel.from_pretrained(model_name="unsloth/gemma-4-E4B-it", dtype=None, max_seq_length=3072,
                                             load_in_4bit=True, full_finetuning=False)
model = FastModel.get_peft_model(model, finetune_vision_layers=False, finetune_language_layers=True,
                                 finetune_attention_modules=True, finetune_mlp_modules=True,
                                 r=32, lora_alpha=64, lora_dropout=0.05, bias="none", random_state=3407)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
ds = Dataset.from_list(examples)
ds = ds.map(lambda b: {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in b["messages"]]}, batched=True)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, eval_dataset=None,
                     args=SFTConfig(dataset_text_field="text", per_device_train_batch_size=4, gradient_accumulation_steps=4,
                                    warmup_steps=5, num_train_epochs=2, learning_rate=1e-4, logging_steps=1,
                                    optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear", seed=3407,
                                    max_length=3072, report_to="none"))
trainer = train_on_responses_only(trainer)
masked_all = all(all(x == -100 for x in ex["labels"]) for ex in trainer.train_dataset.select(range(min(8, len(trainer.train_dataset)))))
BOS_ID = getattr(tokenizer, "tokenizer", tokenizer).bos_token_id
bos_count = sum(1 for x in trainer.train_dataset[0]["input_ids"] if x == BOS_ID)   # 1 つだけのはず
stats = trainer.train()
losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
lora_b = sum(float(p.detach().abs().sum()) for n, p in model.named_parameters() if "lora_B" in n)
train_sec = time.time() - t
TRAIN_TEXT = ds[0]["text"]
print("TRAIN_JSON " + json.dumps({"examples": len(examples), "trainable": trainable, "masked_all": masked_all, "bos_count": bos_count,
                                  "loss_first": losses[0] if losses else None, "loss_last": losses[-1] if losses else None,
                                  "lora_b_abs_sum": lora_b, "seconds": round(train_sec),
                                  "peak_gib": round(torch.cuda.max_memory_reserved() / 2**30, 2)}, ensure_ascii=False))
stage("学習", "pass" if trainable and not masked_all and bos_count == 1 and lora_b > 0 and losses and losses[0] == losses[0] else "prep_failure",
      seconds=round(train_sec), loss_first=losses[0] if losses else None)

In [ ]:
# 結合 → bf16 GGUF → Q4_K_M。対照（学習していない E4B）も同じ手順
t = time.time()
model.save_pretrained_merged("/content/merged16", tokenizer)
merge_sec = time.time() - t
del model, trainer; import gc; gc.collect(); torch.cuda.empty_cache()
from huggingface_hub import snapshot_download
BASE_DIR = snapshot_download("unsloth/gemma-4-E4B-it", local_dir="/content/base16")

def to_q4(src, name):
    t = time.time()
    r1 = subprocess.run(["/content/conv-env/bin/python", "/content/llama.cpp/convert_hf_to_gguf.py", src, "--outtype", "bf16",
                         "--outfile", f"/content/{name}-bf16.gguf"], capture_output=True, text=True)
    if r1.returncode != 0:
        print(r1.stdout[-1500:], r1.stderr[-3000:]); raise RuntimeError(f"{name} の変換に失敗")
    r2 = subprocess.run([QUANTIZE, f"/content/{name}-bf16.gguf", f"/content/{name}-q4km.gguf", "Q4_K_M"],
                        capture_output=True, text=True, env=LLAMA_ENV)
    if r2.returncode != 0:
        print(r2.stdout[-1500:], r2.stderr[-1500:]); raise RuntimeError(f"{name} の量子化に失敗")
    os.remove(f"/content/{name}-bf16.gguf")   # 場所を空ける
    return round(time.time() - t), round(os.path.getsize(f"/content/{name}-q4km.gguf") / 2**30, 2)

conv_student = to_q4("/content/merged16", "student")
conv_control = to_q4(BASE_DIR, "control")
# 結合後の重みが元と違うか（同じ名前の層を 1 つ比べる）
from safetensors import safe_open
def first_tensor(d, key_part):
    for f in sorted(pathlib.Path(d).glob("*.safetensors")):
        with safe_open(str(f), "pt") as s:
            for k in s.keys():
                if key_part in k:
                    return k, s.get_tensor(k).float()
k1, w1 = first_tensor("/content/merged16", "layers.0.self_attn.q_proj.weight")
k2, w2 = first_tensor(BASE_DIR, "layers.0.self_attn.q_proj.weight")
diff = float((w1 - w2).abs().max()) if w1.shape == w2.shape else None
print("CONVERT_JSON " + json.dumps({"merge_sec": round(merge_sec), "student": conv_student, "control": conv_control,
                                    "keys": [k1, k2], "max_abs_diff": diff}, ensure_ascii=False))
stage("結合と変換", "pass" if diff else "prep_failure", merge_sec=round(merge_sec), student=conv_student, control=conv_control, diff=diff)

In [ ]:
# 守りの問 40 問（対照と学習後）と、会話の形の一致
GUARD_OUT = {}
for name in ("control", "student"):
    sp = launch([SERVER, "-m", f"/content/{name}-q4km.gguf", "-ngl", "999", "-c", "4096", "-np", "1", "--jinja",
                 "--reasoning", "off", "--host", "127.0.0.1", "--port", "8002"], OUTP / f"{name}.log")
    if not wait_up(8002, sp):
        print("\n".join(open(OUTP / f"{name}.log").read().splitlines()[-30:])); raise RuntimeError(f"{name} が起動しない")
    if name == "student":
        req = urllib.request.Request("http://127.0.0.1:8002/apply-template",
                                     json.dumps({"messages": examples[0]["messages"][:-1]}).encode(), {"Content-Type": "application/json"})
        server_prompt = json.loads(urllib.request.urlopen(req).read())["prompt"]
        # BOS を両方から外してから、サーバーの文が学習文の頭と一字一句同じかを見る（11/02 の試しは外す前に切っていた）
        sp_body, tr_body = server_prompt.removeprefix("<bos>"), TRAIN_TEXT.removeprefix("<bos>")
        TEMPLATE_MATCH = {"equal": tr_body.startswith(sp_body), "server_head": sp_body[:120], "train_head": tr_body[:120]}
        print("TEMPLATE_JSON " + json.dumps(TEMPLATE_MATCH, ensure_ascii=False))
    res = []
    for g in GUARD:
        out = chat(8002, g["messages"], 0.1, 1024, stream=True)
        res.append({"id": g["id"], "guard": g["guard"], "pass": guardmod.judge(g, out["content"]),
                    "first": out["first"], "seconds": out["seconds"], "answer": out["content"]})
    kill(sp)
    GUARD_OUT[name] = res
    by = Counter(); ok = Counter()
    for x in res:
        by[x["guard"]] += 1; ok[x["guard"]] += x["pass"]
    print(f"GUARD_{name.upper()}_JSON " + json.dumps({"pass": {k: f"{ok[k]}/{by[k]}" for k in by},
          "seconds_median": round(statistics.median(x["seconds"] for x in res), 2)}, ensure_ascii=False))
stage("守りの問", "pass")

In [ ]:
# まとめ（時間だけ。成績は使わない）
SUMMARY = {"repo": REPO_SHA, "gpu": GPU, "stages": STAGES, "teacher": summary,
           "total_minutes": round((time.time() - T0) / 60, 1)}
print("SUMMARY_JSON " + json.dumps(SUMMARY, ensure_ascii=False))